# SML & BDA Lab Experiment 7
## Title: Machine Learning Model Packaging, Serialization & Deployment using FastAPI and Uvicorn
### Name: Manan Chahal
### Roll No: 23102C0044

---

### Objectives:
1. Load and explore the classic Iris flower dataset.
2. Build an end-to-end Machine Learning classification pipeline with feature scaling (`StandardScaler`) and ensemble learning (`RandomForestClassifier`).
3. Evaluate model generalization on an independent test dataset using Accuracy, Classification Report, and Confusion Matrix.
4. Serialize and package the trained pipeline and metadata into a self-contained pickle artifact (`model.pkl`).
5. Deserialize the artifact and verify sanity test inferences on flower species samples.
6. Serve real-time predictions by building a REST API with **FastAPI** (`main.py`) providing interactive Swagger UI docs.
7. Launch the **Uvicorn** ASGI server in the final cell to serve predictions on `http://127.0.0.1:8000`.

## 1. Environment Setup & Library Imports
Importing necessary modules for data processing, machine learning modeling, serialization, and ASGI serving.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Environment configured and libraries imported successfully.")

Environment configured and libraries imported successfully.


## 2. Dataset Loading & Exploratory Data Analysis (EDA)
Loading the Iris dataset from `sklearn.datasets` and formatting feature names into API-friendly standard identifiers (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`).

In [2]:
# Load Iris dataset
iris = load_iris(as_frame=True)

# Standardize column names to API-friendly snake_case
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df = iris.data.copy()
df.columns = feature_names
df['species_id'] = iris.target
df['species_name'] = df['species_id'].map(lambda x: iris.target_names[x])

print("Dataset Shape:", df.shape)
print("\nTarget Species Mapping:")
for i, name in enumerate(iris.target_names):
    print(f"  Class {i} -> {name}")

print("\nFirst 5 Records:")
df.head()

Dataset Shape: (150, 6)

Target Species Mapping:
  Class 0 -> setosa
  Class 1 -> versicolor
  Class 2 -> virginica

First 5 Records:


,sepal_length,sepal_width,petal_length,petal_width,species_id,species_name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


## 3. Class Balance and Summary Statistics
Inspecting class distributions and descriptive statistics across the morphological features.

In [3]:
print("Class Distribution:")
print(df['species_name'].value_counts())

print("\nSummary Statistics:")
df[feature_names].describe()

Class Distribution:
species_name
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

Summary Statistics:


,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


## 4. Train-Test Split
Splitting into an 80% training set and 20% test set using stratified sampling to preserve class balance.

In [4]:
X = df[feature_names]
y = df['species_id']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size:  {X_test.shape[0]} samples")

Training set size: 120 samples
Testing set size:  30 samples


## 5. Model Pipeline Construction & Training
Constructing a Scikit-Learn `Pipeline` bundling `StandardScaler` with `RandomForestClassifier`.
Using a unified Pipeline encapsulates preprocessing inside the model artifact, preventing data leakage and eliminating the need for independent preprocessor files during API inference.

In [5]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42))
])

pipeline.fit(X_train, y_train)
print("Pipeline trained successfully!")
print("Pipeline Steps:")
for name, step in pipeline.named_steps.items():
    print(f"  {name}: {step}")

Pipeline trained successfully!
Pipeline Steps:
  scaler: StandardScaler()
  classifier: RandomForestClassifier(max_depth=4, random_state=42)


## 6. Model Evaluation
Evaluating the model on the test dataset across Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.

In [6]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Overall Accuracy: {acc:.4f} ({acc * 100:.2f}%)\n")
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=iris.target_names))
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=iris.target_names, columns=[f"Pred_{n}" for n in iris.target_names])
print(cm_df)

Overall Accuracy: 0.9667 (96.67%)

Classification Report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Confusion Matrix:
            Pred_setosa  Pred_versicolor  Pred_virginica
setosa               10                0               0
versicolor            0                9               1
virginica             0                0              10


## 7. Model Serialization & Packaging (`model.pkl`)
Packaging the trained Pipeline along with feature names, target classes, and descriptive metadata into a single pickle artifact file (`model.pkl`).

In [7]:
artifact = {
    'model': pipeline,
    'feature_names': feature_names,
    'target_names': list(iris.target_names),
    'description': 'Random Forest Classifier for Iris flower species classification trained on Iris dataset'
}

model_filename = 'model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(artifact, f)

file_size_kb = os.path.getsize(model_filename) / 1024
print(f"Artifact successfully serialized to {model_filename}")
print(f"File Size: {file_size_kb:.2f} KB")

Artifact successfully serialized to model.pkl
File Size: 137.80 KB


## 8. Deserialization & Sample Inference Sanity Check
Testing the saved `model.pkl` to confirm that predictions and probability distributions match expectations for all three species classes.

In [8]:
# Load the saved model artifact
with open('model.pkl', 'rb') as f:
    loaded_artifact = pickle.load(f)

loaded_model = loaded_artifact['model']
target_classes = loaded_artifact['target_names']

# Define 3 representative test samples
test_cases = pd.DataFrame([
    {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}, # Setosa
    {'sepal_length': 6.0, 'sepal_width': 2.9, 'petal_length': 4.5, 'petal_width': 1.5}, # Versicolor
    {'sepal_length': 6.7, 'sepal_width': 3.0, 'petal_length': 5.2, 'petal_width': 2.3}  # Virginica
], columns=loaded_artifact['feature_names'])

predictions = loaded_model.predict(test_cases)
probabilities = loaded_model.predict_proba(test_cases)

for i, (pred, p_dist) in enumerate(zip(predictions, probabilities)):
    pred_species = target_classes[pred]
    confidence = p_dist[pred] * 100
    print(f"Sample {i+1}: Features = {test_cases.iloc[i].to_dict()} -> Predicted: {pred_species.upper()} (Confidence: {confidence:.2f}%)")

Sample 1: Features = {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2} -> Predicted: SETOSA (Confidence: 100.00%)
Sample 2: Features = {'sepal_length': 6.0, 'sepal_width': 2.9, 'petal_length': 4.5, 'petal_width': 1.5} -> Predicted: VERSICOLOR (Confidence: 97.78%)
Sample 3: Features = {'sepal_length': 6.7, 'sepal_width': 3.0, 'petal_length': 5.2, 'petal_width': 2.3} -> Predicted: VIRGINICA (Confidence: 99.72%)


## 9. Launch FastAPI Serving Endpoint via Uvicorn
The serialized `model.pkl` is loaded by `main.py` which provides:
* **Interactive Swagger UI**: `http://127.0.0.1:8000/docs`
* **Alternative ReDoc UI**: `http://127.0.0.1:8000/redoc`
* **Single Prediction**: `POST http://127.0.0.1:8000/predict`
* **Batch Prediction**: `POST http://127.0.0.1:8000/predict/batch`
* **Health Check**: `GET http://127.0.0.1:8000/health`
* **Model Information**: `GET http://127.0.0.1:8000/model-info`

Executing the cell below runs the Uvicorn ASGI server directly from the notebook.
*(Note: To stop the server when finished, click the Stop/Interrupt Kernel button in your notebook toolbar).*